In [3]:

import glob
import io
import os
from pprint import pprint
import zipfile
from PIL import Image
import requests, json, mimetypes
import matplotlib.pyplot as plt
from settings import BASE_DIR
%matplotlib inline
URL = "http://localhost:8008/main/translate"

def run_translate_inpaint(manga_name, chapter_number, original_image_paths, batch_size = 5):
    print(manga_name, chapter_number, original_image_paths)
    page_numbers = [int(path.split("/")[-1].split(".")[0]) for path in original_image_paths]
    # Sử dụng ExitStack để quản lý file handles
    from contextlib import ExitStack

    total_pages = len(original_image_paths)
    total_batches = (total_pages + batch_size - 1) // batch_size
    print(f"Tổng số trang: {total_pages}, chia thành {total_batches} batch")

    all_results = []
    
    for i in range(0, total_pages, batch_size):
        batch_paths = original_image_paths[i:i+batch_size]
        batch_numbers = page_numbers[i:i+batch_size]
        current_batch = i // batch_size + 1
        
        print(f"\nĐang xử lý batch {current_batch}/{total_batches} ({len(batch_paths)} trang)")
        
        # Tạo form data cho batch này
        batch_form_data = [
            ("target_lang", "vietnamese"),
            ("story_name", str(manga_name)),
            ("chapter_number", str(chapter_number)),
        ]
        print(batch_form_data)
        
        for n in batch_numbers:
            batch_form_data.append(("page_numbers", str(n)))
        
        # Sử dụng ExitStack để quản lý file handles
        with ExitStack() as stack:
            batch_files = [
                ("page_images", 
                (path.split("/")[-1], 
                stack.enter_context(open(path, "rb")), "image/jpeg")
                )
                for path in batch_paths
            ]
            
            # Thiết lập timeout dài hơn (5 phút)
            timeout = 3000
            
            # Gửi request cho batch hiện tại
            response = requests.post(URL, data=batch_form_data, files=batch_files, timeout=timeout)
            
            if response.status_code == 200:
                data = response.json()
                print(f"page_number: {data['page_number']}")
                for item in data['outputs']:
                    for output in item:
                        print(f"source_text: {output['source_text']}")
                        print(f"translation: {output['translation']}")
                        print("-" * 5)
                    print("-" * 10)
            else:
                print(f"Lỗi batch {current_batch}: {response.status_code}")
                print(response.text)


In [4]:
chapter_numbers = [133, 134, 135, 136, 137]
for chapter_number in chapter_numbers:
    manga_name = "Yule"
    folder_path = f"/home/aorus/workspaces/test_commic/data/{manga_name}"
    chapter_folder_path = os.path.join(folder_path, "Raw", str(chapter_number))
    original_image_paths = glob.glob(os.path.join(chapter_folder_path, "*.jpg"))
    run_translate_inpaint(manga_name, chapter_number, original_image_paths, batch_size=10)

Yule 133 ['/home/aorus/workspaces/test_commic/data/Yule/Raw/133/26.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/17.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/11.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/23.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/7.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/22.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/18.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/25.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/3.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/19.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/9.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/12.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/13.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/5.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw/133/20.jpg', '/home/aorus/workspaces/test_commic/data/Yule/Raw

KeyboardInterrupt: 

In [9]:
CHAPTER_NUMBER = 133
PAGE_NUMBER = 9
MANGA_NAME = "Yule"
folder_path = f"{BASE_DIR}/test_commic/data/{MANGA_NAME}"
chapter_folder_path = os.path.join(folder_path, "Raw", str(CHAPTER_NUMBER))
original_image_paths = [f"{BASE_DIR}/test_commic/data/Yule/Raw/{CHAPTER_NUMBER}/{PAGE_NUMBER}.jpg"]
run_translate_inpaint(MANGA_NAME, CHAPTER_NUMBER, original_image_paths, batch_size=11)


Yule 133 ['/home/aorus/workspaces/magiv2/test_commic/data/Yule/Raw/133/9.jpg']
Tổng số trang: 1, chia thành 1 batch

Đang xử lý batch 1/1 (1 trang)
[('target_lang', 'vietnamese'), ('story_name', 'Yule'), ('chapter_number', '133')]


page_number: 9
source_text: Don't need that!!!
translation: Không cần đâu!
-----
source_text: You're really clueless.
translation: Ngươi thật sự ngây ngô.
-----
source_text: Cluffless about what? Is it fun? I'll take it instead!
translation: Ngây ngô chuyện gì? Chẳng lẽ có chuyện thú vị? Ta đây xin thử một phen!
-----
source_text: As expected of the Townspeople living under the Yuanmo Sept, they're all weird.
translation: Quả nhiên là dân chúng dưới trướng Nguyên ma tông, đều là những kẻ kỳ dị.
-----
source_text: We don't want whatever you're offering...
translation: Chúng ta không cần bất cứ thứ gì ngươi ban tặng…
-----
----------
